# Project Personal Chat Assistant

By Akshat Kesharwani | Free tools only (Google Colab + free-tier APIs)

## Problem Statement

A generic chatbot doesn't sound like you -- it replies the same flat way to everyone. Personalization systems fix this by fine-tuning on someone's own chat style. Here we fine-tune GPT-2 on everyday conversation data, and add a simple 'style profile' text you can edit to steer the tone without retraining the whole model.

### SECTION 1: Install & Import Libraries

In [1]:
!pip install -q transformers datasets accelerate openai

import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)

# ---- Small AI helper: call Groq (free) ----
# Only using Groq here -- it's free, fast, and just needs an API key,
# no extra setup like Ollama needs.
# Get a free key at https://console.groq.com/keys
# (Update: as of June 2026 Groq retired the old Llama-3.3-70B model,
# the current one to use is "openai/gpt-oss-120b".)
from getpass import getpass
from openai import OpenAI

groq_key = getpass("Enter your Groq API key: ")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

def call_llm(prompt, max_tokens=300):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

Enter your Groq API key: ··········


### SECTION 2: Load Personal-Style Chat Dataset (with robust fallback)

In [2]:
# heads up: "daily_dialog" uses the old loading-script way of doing things,
# which newer `datasets` versions don't allow by default. so we try a
# couple of repo names, and if none of those work either, we just fall
# back to some chat pairs we wrote ourselves. DailyDialog (or the
# fallback) is a free stand-in for "personal conversations" -- swap in
# your own exported chat history (WhatsApp/Telegram export -> CSV) for a
# genuinely personalized assistant using the same pipeline.
def robust_load_dataset(repo_candidates, split="train"):
    for repo_id in repo_candidates:
        try:
            print(f"[dataset] Trying load_dataset('{repo_id}', split='{split}')...")
            ds = load_dataset(repo_id, split=split)
            print(f"[dataset] Loaded '{repo_id}' successfully ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] '{repo_id}' standard load failed: {type(e).__name__}: {e}")
        try:
            parquet_path = f"hf://datasets/{repo_id}@refs/convert/parquet/default/{split}/*.parquet"
            print(f"[dataset] Trying Parquet fallback: {parquet_path}")
            ds = load_dataset("parquet", data_files=parquet_path, split="train")
            print(f"[dataset] Parquet fallback for '{repo_id}' succeeded ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] Parquet fallback for '{repo_id}' failed: {type(e).__name__}: {e}")
    return None

raw_train = robust_load_dataset(["li2017dailydialog/daily_dialog", "daily_dialog", "roskoN/dailydialog"], split="train")

pairs = []
if raw_train is not None:
    all_dialogs = raw_train["dialog"]
    # flatten multi-turn dialogues into (context -> next utterance) pairs
    for dialog in all_dialogs:
        for i in range(len(dialog) - 1):
            pairs.append({"context": dialog[i].strip(), "response": dialog[i + 1].strip()})

if not pairs:
    # ---- Synthetic offline fallback -- guarantees the notebook always runs ----
    SYNTHETIC_CHATS = [
        {"context": "Hey, are we still on for tomorrow?", "response": "Yep, still on! What time works for you?"},
        {"context": "I'm running about 10 minutes late.", "response": "No worries, take your time, I'll grab us a table."},
        {"context": "Did you see the game last night?", "response": "Yes! That last-minute goal was insane."},
        {"context": "Can you send me the notes from the meeting?", "response": "Sure, sending them over right now."},
        {"context": "I'm thinking of ordering pizza tonight.", "response": "Sounds good, get me the usual pepperoni please."},
        {"context": "How was your weekend?", "response": "Pretty relaxed, mostly just caught up on sleep and read a bit."},
        {"context": "Do you want to grab coffee this weekend?", "response": "Definitely, Saturday morning works best for me."},
        {"context": "I finally finished that book you recommended.", "response": "No way, what did you think of the ending?"},
        {"context": "Can you help me move this weekend?", "response": "Of course, just tell me what time and I'll be there."},
        {"context": "I think I left my charger at your place.", "response": "Oh yeah, I found it! I'll bring it when I see you next."},
    ]
    pairs = SYNTHETIC_CHATS * 60  # replicate so the Trainer has a workable number of steps
    print("[dataset] Using synthetic fallback: swap in a real Kaggle/HF dataset or your own chat export before reporting final metrics.")

df = pd.DataFrame(pairs)
df = df.sample(n=min(50000, len(df)), random_state=42, replace=len(df) < 50000).reset_index(drop=True)
print("Chat message pairs:", len(df))

[dataset] Trying load_dataset('li2017dailydialog/daily_dialog', split='train')...


README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

daily_dialog.py:   0%|          | 0.00/4.85k [00:00<?, ?B/s]

[dataset] 'li2017dailydialog/daily_dialog' standard load failed: RuntimeError: Dataset scripts are no longer supported, but found daily_dialog.py
[dataset] Trying Parquet fallback: hf://datasets/li2017dailydialog/daily_dialog@refs/convert/parquet/default/train/*.parquet


default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.61MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

[dataset] Parquet fallback for 'li2017dailydialog/daily_dialog' succeeded (11118 rows).
Chat message pairs: 50000


### SECTION 3: Format for Causal LM Fine-Tuning

In [3]:
df["text"] = "User: " + df["context"] + "\nMe: " + df["response"]
dataset = Dataset.from_pandas(df[["text"]]).train_test_split(test_size=0.05, seed=42)

### SECTION 4: Tokenize

In [4]:
model_checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    texts = [t + tokenizer.eos_token for t in batch["text"]]
    return tokenizer(texts, truncation=True, max_length=96, padding="max_length")

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/47500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

### SECTION 5: Load & Fine-Tune GPT-2-small

In [5]:
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

use_fp16 = torch.cuda.is_available()  # fp16 only works with a real GPU -- Runtime > Change runtime type > T4 GPU
print("GPU available:", use_fp16)

training_args = TrainingArguments(
    output_dir="./personal-chat-checkpoints",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,      # kept small so free Colab doesn't run out of RAM
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    gradient_accumulation_steps=4,   # makes up for the smaller batch size above
    gradient_checkpointing=True,     # uses less memory, trains a little slower
    save_strategy="epoch",
    save_total_limit=1,
    fp16=use_fp16,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
    data_collator=data_collator
)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPU available: True


...

Epoch,Training Loss,Validation Loss
1,2.529538,2.422320
2,2.337770,2.349561
3,2.225263,2.329537


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8907, training_loss=2.397929291699997, metrics={'train_runtime': 3172.4459, 'train_samples_per_second': 44.918, 'train_steps_per_second': 2.808, 'total_flos': 6981396480000000.0, 'train_loss': 2.397929291699997, 'epoch': 3.0})

### SECTION 6: Personalization via a Style Profile

In [6]:
# Lightweight personalization layer: a short user "style profile"
# (tone, common phrases) is prepended to every prompt so the same
# fine-tuned base model reflects a specific person's chat style.
STYLE_PROFILE = "Style: casual, friendly, uses short sentences and emojis occasionally."

def chat_response(user_message, max_new_tokens=40):
    prompt = f"{STYLE_PROFILE}\nUser: {user_message}\nMe:"
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **{k: v.to(model.device) for k, v in inputs.items()},
            max_new_tokens=max_new_tokens, do_sample=True, temperature=0.85, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    reply = tokenizer.decode(output[0], skip_special_tokens=True)
    return reply.split("Me:")[-1].strip()

print(chat_response("Hey, are we still on for tomorrow?"))

I ’ m just getting started . Let ’ s just say you ’ Ve got a lot of things to learn . We should just get together and have fun . Let ’ s go


### SECTION 7: Reduce Repetitive Usage -- Perplexity Check

In [7]:
# A quick perplexity measurement on held-out text -- lower perplexity
# vs. the base (untuned) GPT-2 shows the model learned this chat style,
# meaning users get relevant replies faster with fewer re-prompts.
import math

def compute_perplexity(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=96)
    with torch.no_grad():
        outputs = model(**{k: v.to(model.device) for k, v in inputs.items()}, labels=inputs["input_ids"].to(model.device))
    return math.exp(outputs.loss.item())

sample_texts = dataset["test"]["text"][:20] if "text" in dataset["test"].column_names else df["text"].sample(20).tolist()
avg_perplexity = sum(compute_perplexity(t) for t in sample_texts) / len(sample_texts)
print(f"Average perplexity on held-out chat text: {avg_perplexity:.2f}")

Average perplexity on held-out chat text: 11.57


### SECTION 8: Save Model

In [8]:
model.save_pretrained("./personal-chat-assistant")
tokenizer.save_pretrained("./personal-chat-assistant")
print("Model saved to ./personal-chat-assistant")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./personal-chat-assistant
